# Airborne lidar validation of v3.1 vegetation height (LVIS 2017/2019, G-LiHT)

Companion to `validation_v3.1_height_agbd.ipynb` (field plots). Airborne lidar adds what plots cannot:

* **Same-year references.** ABoVE LVIS flew in 2017 and 2019, both v3.1 map years, so there is no temporal gap and no stand-age screening is needed for the same-year comparison.
* **The same height definition.** LVIS L3 gridded `RH098_mean_30m` is a 98th-percentile waveform height, the quantity the map is trained toward (ATL08 canopy height). Field plots use Lorey's, median, or top height.
* **Change validation.** Pixels flown in both 2017 and 2019 test whether the map's 2017→2019 differences track real change (loss) rather than noise.
* **Pixel-perfect aggregation (G-LiHT).** 1 m CHMs are aggregated *on the map's own 30 m grid*, which removes the grid-misregistration noise that point sampling of LVIS carries.

What existed before in this repo: `examine_height_boreal_lvis.ipynb` compared an older 2020 gridded ATL08 height (a collaborator's 20 M-pixel CSV, not v3.1). `ht_validation_updated.ipynb` is an unfinished extraction loop for v1.5 maps. `above_lidar_check.ipynb` cross-checked LVIS against G-LiHT (2014/2018) from local files. No v3.1 lidar validation existed.

Runs on MAAP (`pangeo` env). Needs Earthdata login for ORNL DAAC (LVIS) and LP DAAC (G-LiHT) S3 credentials.

--------------------------------------------------------------------------------
Paul Montesano, PhD | NASA GSFC, ESSIC
--------------------------------------------------------------------------------

In [ ]:
import os, re, glob, json, warnings, datetime, zlib
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterio.warp
import rasterio.windows
from rasterio.windows import Window
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.session import AWSSession
from shapely.geometry import box
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=RuntimeWarning)

## 1. Configuration

In [ ]:
MAP_YEARS = list(range(2016, 2026))
# v3.1 height tiles (same footprints used by extract_rasters_to_val_plots.ipynb); bands: 1 = mean_ht, 2 = std_ht
HT_FOOTPRINT_FN = '/projects/my-public-bucket/databank/footprints/footprints_maap_Ht_H30_{YEAR}_v3.1_multiyr-s3.gpkg'
# CACC 2020 (bands: ..., 9 = age_mean, 10 = age_std)
CACC_FOOTPRINT_FN = '/projects/my-public-bucket/DPS_tile_lists/CACC/carbon_accumulation_v3/CACC_2020_v3.1_multiyr_nsims050/CACC_tindex_master.gpkg'
CACC_AGE_BANDS = (9, 10)

# LVIS L3 gridded (ORNL DAAC, ABoVE_LVIS_VegetationStructure_1923)
LVIS_BUCKET = 'ornl-cumulus-prod-protected/above/ABoVE_LVIS_VegetationStructure/data'
LVIS_HT = 'RH098_mean_30m'
LVIS_CC = 'CC_gte_01p37m_30m'      # canopy cover >= 1.37 m [%, stored x100 in some files -> checked below]
N_PER_TILE = 150                   # random pixels per LVIS tile (after thinning)
THIN_STRIDE = 3                    # keep pixels on a 3x3 lattice (90 m) to reduce spatial autocorrelation
MAP_WINDOW = 1                     # 1 = nearest map pixel; 3 = 3x3 mean on both sides (absorbs grid misregistration)

# G-LiHT 1 m CHM (LP DAAC). Short name to VERIFY with the discovery cell below.
GLIHT_SHORT_NAME = 'GLCHMT'
GLIHT_CHM_PATTERN = r'CHM.*\.tif$'
GLIHT_BBOX = (-170.0, 50.0, -100.0, 72.0)    # ABoVE core domain (lon/lat)
GLIHT_TEMPORAL = ('2016-01-01', '2025-12-31')
GLIHT_MAX_GRANULES = 60
GLIHT_MIN_VALID_FRAC = 0.8          # min fraction of valid 1 m cells within a 30 m map pixel
GLIHT_HT_PCT = 98                   # percentile of 1 m CHM within each 30 m pixel (comparable to RH98)

OUT_DIR = os.environ.get('VAL_OUT_DIR', f'/projects/my-private-bucket/validation_v3.1/{datetime.date.today():%Y%m%d}_lidar')
os.makedirs(OUT_DIR, exist_ok=True)
RNG = np.random.default_rng(42)

PAL = {'blue': '#2a78d6', 'orange': '#eb6834', 'aqua': '#1baf7a', 'yellow': '#eda100', 'grey': '#8c8c8c'}
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.color': '#e6e6e6', 'axes.spines.top': False, 'axes.spines.right': False})

## 2. Helpers
Same metric definitions as the field-plot notebook: errors are against the 1:1 line (bias = mean(map − ref), RMSD), **not** OLS intercept / residual SE.

In [ ]:
def val_metrics(pred, ref):
    pred, ref = np.asarray(pred, float), np.asarray(ref, float)
    ok = np.isfinite(pred) & np.isfinite(ref)
    pred, ref = pred[ok], ref[ok]
    n = len(ref)
    if n < 3:
        return pd.Series({'n': n})
    d = pred - ref
    slope, intercept = np.polyfit(ref, pred, 1) if np.ptp(ref) > 0 else (np.nan, np.nan)
    return pd.Series({'n': n, 'mean_ref': ref.mean(), 'bias': d.mean(), 'rbias': 100 * d.mean() / ref.mean(),
                      'rmsd': np.sqrt((d ** 2).mean()), 'rrmsd': 100 * np.sqrt((d ** 2).mean()) / ref.mean(),
                      'mae': np.abs(d).mean(), 'r2': np.corrcoef(pred, ref)[0, 1] ** 2 if np.std(pred) > 0 else np.nan,
                      'slope': slope, 'intercept': intercept})

def boot_ci(df, cluster='cluster_id', B=500):
    """Cluster bootstrap (resample flightline tiles / granules) for bias and RMSD."""
    groups = [g for _, g in df.groupby(cluster)]
    if len(groups) < 5:
        return pd.Series({'bias_lo': np.nan, 'bias_hi': np.nan, 'rmsd_lo': np.nan, 'rmsd_hi': np.nan, 'n_clusters': len(groups)})
    d_list = [(g.pred - g.ref).to_numpy() for g in groups]
    stats = []
    for _ in range(B):
        d = np.concatenate([d_list[i] for i in RNG.integers(0, len(d_list), len(d_list))])
        stats.append((d.mean(), np.sqrt((d ** 2).mean())))
    s = np.array(stats)
    return pd.Series({'bias_lo': np.percentile(s[:, 0], 2.5), 'bias_hi': np.percentile(s[:, 0], 97.5),
                      'rmsd_lo': np.percentile(s[:, 1], 2.5), 'rmsd_hi': np.percentile(s[:, 1], 97.5), 'n_clusters': len(groups)})

def summarise(df, by, ci=True):
    rows = []
    for k, g in df.groupby(by, observed=True):
        m = val_metrics(g.pred, g.ref)
        if ci and m['n'] >= 3:
            m = pd.concat([m, boot_ci(g)])
        rows.append(pd.concat([pd.Series(dict(zip(by if isinstance(by, list) else [by], k if isinstance(k, tuple) else (k,)))), m]))
    return pd.DataFrame(rows)

def hexval(ax, ref, pred, lim, title, unit='m'):
    ok = np.isfinite(ref) & np.isfinite(pred)
    hb = ax.hexbin(ref[ok], pred[ok], gridsize=50, extent=(0, lim, 0, lim), mincnt=1, bins='log', cmap='Blues')
    ax.plot([0, lim], [0, lim], ls='--', c='0.3', lw=1)
    m = val_metrics(pred, ref)
    if m['n'] >= 3:
        ax.text(0.03, 0.97, f"n = {int(m['n']):,}\nbias = {m['bias']:+.1f} {unit}\nRMSD = {m['rmsd']:.1f} {unit} ({m['rrmsd']:.0f}%)\nR² = {m['r2']:.2f}",
                transform=ax.transAxes, va='top', fontsize=8, bbox=dict(fc='white', ec='none', alpha=0.85))
    ax.set(xlim=(0, lim), ylim=(0, lim), aspect='equal', title=title)
    return hb

In [ ]:
def sample_raster(path, xs, ys, pts_crs, bands=(1,), win=1, session=None):
    """Value(s) at points; win>1 returns the nanmean of a win x win block centred on each point."""
    env = rasterio.Env(session=session) if session is not None else rasterio.Env()
    with env, rasterio.open(path) as src:
        if str(src.crs) != str(pts_crs):
            from pyproj import Transformer
            xs, ys = Transformer.from_crs(pts_crs, src.crs, always_xy=True).transform(np.asarray(xs), np.asarray(ys))
        nod = src.nodata
        out = np.full((len(xs), len(bands)), np.nan)
        if win == 1:
            for i, v in enumerate(src.sample(zip(xs, ys), indexes=list(bands))):
                out[i] = v
        else:
            h = win // 2
            for i, (x, y) in enumerate(zip(xs, ys)):
                r, c = src.index(x, y)
                a = src.read(list(bands), window=Window(c - h, r - h, win, win), boundless=True, fill_value=np.nan if nod is None else nod).astype(float)
                if nod is not None:
                    a[a == nod] = np.nan
                out[i] = np.nanmean(a.reshape(len(bands), -1), axis=1)
        if nod is not None:
            out[out == nod] = np.nan
    return out

def extract_map(points, footprint_fn, bands, names, win=MAP_WINDOW):
    """points: GeoDataFrame with columns x, y in points.crs. Uses the footprint index to find each point's map tile."""
    fp = gpd.read_file(footprint_fn)
    path_col = 's3_path' if 's3_path' in fp.columns else [c for c in fp.columns if 'path' in c.lower()][0]
    pts = points[['geometry']].to_crs(fp.crs)
    j = gpd.sjoin(pts, fp[[path_col, 'geometry']], how='left', predicate='within')
    j = j[~j.index.duplicated()]
    out = pd.DataFrame(np.nan, index=points.index, columns=names)
    def one(item):
        path, idx = item
        sub = points.loc[idx]
        return idx, sample_raster(path, sub.geometry.x.values, sub.geometry.y.values, points.crs, bands, win)
    groups = list(j.dropna(subset=[path_col]).groupby(path_col).groups.items())
    with ThreadPoolExecutor(8) as ex:
        for idx, vals in ex.map(one, groups):
            out.loc[idx, names] = vals
    return out

## 3. LVIS: access and sampling
Credentials: ORNL DAAC temporary S3 credentials (as in `s3_data_access_LVIS.ipynb`). They are passed only to LVIS reads through an `AWSSession`, so MAAP-bucket reads keep the platform role.

In [ ]:
def ornl_session():
    import requests
    try:
        from maap.maap import MAAP
        c = MAAP().aws.earthdata_s3_credentials('https://data.ornldaac.earthdata.nasa.gov/s3credentials')
    except Exception:
        c = requests.get('https://data.ornldaac.earthdata.nasa.gov/s3credentials').json()
    return c, AWSSession(aws_access_key_id=c['accessKeyId'], aws_secret_access_key=c['secretAccessKey'], aws_session_token=c['sessionToken'], region_name='us-west-2')

def list_lvis(creds):
    import s3fs
    fs = s3fs.S3FileSystem(key=creds['accessKeyId'], secret=creds['secretAccessKey'], token=creds['sessionToken'])
    files = ['s3://' + f for f in fs.glob(f'{LVIS_BUCKET}/*{LVIS_HT}.tif')]
    df = pd.DataFrame({'ht_path': files})
    df['year'] = df.ht_path.str.extract(r'ABoVE(\d{4})_').astype(int)
    df['tile'] = df.ht_path.str.extract(r'(LVISF\d_ABoVE\d{4}_\d{4}_R\d+_\d+)_')
    df['cc_path'] = df.ht_path.str.replace(LVIS_HT, LVIS_CC, regex=False)
    return df

In [ ]:
def sample_lvis_tile(row, session, n=N_PER_TILE, stride=THIN_STRIDE):
    """Random valid pixels from one LVIS tile, thinned to a stride x stride lattice. Returns records in the tile CRS."""
    with rasterio.Env(session=session) if session is not None else rasterio.Env():
        with rasterio.open(row.ht_path) as src:
            a = src.read(1).astype(float)
            if src.nodata is not None:
                a[a == src.nodata] = np.nan
            a[(a < 0) | (a > 80)] = np.nan
            lat = np.zeros_like(a, bool); lat[::stride, ::stride] = True
            rr, cc = np.where(np.isfinite(a) & lat)
            if len(rr) == 0:
                return None
            rng = np.random.default_rng(zlib.crc32(row.tile.encode()))   # per-tile RNG: reproducible and thread-safe
            k = rng.choice(len(rr), min(n, len(rr)), replace=False)
            rr, cc = rr[k], cc[k]
            xs, ys = rasterio.transform.xy(src.transform, rr, cc)
            crs, bounds = src.crs, src.bounds
        cc_val = np.full(len(rr), np.nan)
        try:
            with rasterio.open(row.cc_path) as c:
                cv = c.read(1).astype(float)
                if c.nodata is not None:
                    cv[cv == c.nodata] = np.nan
                cc_val = cv[rr, cc]
        except Exception:
            pass
    return pd.DataFrame({'x': xs, 'y': ys, 'lvis_ht': a[rr, cc], 'lvis_cc': cc_val, 'year': row.year,
                         'tile': row.tile, 'crs': str(crs)}), (row.tile, row.year, str(crs), box(*bounds))

In [ ]:
RUN_LVIS = True
if RUN_LVIS:
    creds, ornl = ornl_session()
    lvis_files = list_lvis(creds)
    print(lvis_files.groupby('year').size())
    with ThreadPoolExecutor(16) as ex:
        res = [r for r in ex.map(lambda t: sample_lvis_tile(t, ornl), lvis_files.itertuples()) if r is not None]
    lvis_tiles = gpd.GeoDataFrame(
        pd.DataFrame([r[1][:3] for r in res], columns=['tile', 'year', 'crs']),
        geometry=gpd.GeoSeries([r[1][3] for r in res]).values)
    # One CRS per tile: convert each tile's points to lon/lat then to a common equal-area CRS
    parts = []
    for df in (r[0] for r in res):
        g = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x, df.y), crs=df.crs.iloc[0]).to_crs(4326)
        parts.append(g.drop(columns=['crs']))
    lvis = pd.concat(parts, ignore_index=True)
    lvis = gpd.GeoDataFrame(lvis, geometry='geometry', crs=4326)
    # Canopy cover units: some LVIS L3 grids store cover x100
    if np.nanmax(lvis.lvis_cc) > 100:
        lvis['lvis_cc'] = lvis.lvis_cc / 100
    print(f'{len(lvis):,} LVIS pixels from {lvis.tile.nunique()} tiles')

## 4. Extract v3.1 height (all years) and 2020 stand age at LVIS pixels

In [ ]:
if RUN_LVIS:
    for y in MAP_YEARS:
        v = extract_map(lvis, HT_FOOTPRINT_FN.format(YEAR=y), (1, 2), [f'ht_{y}', f'ht_sd_{y}'])
        lvis[[f'ht_{y}', f'ht_sd_{y}']] = v
        print(y, f'{v.iloc[:, 0].notna().mean():.1%} with a map value')
    lvis[['age_map', 'age_map_sd']] = extract_map(lvis, CACC_FOOTPRINT_FN, CACC_AGE_BANDS, ['age_map', 'age_map_sd'])
    lvis.to_parquet(os.path.join(OUT_DIR, 'lvis_samples_v3.1.parquet'))

## 5. Same-year comparison (LVIS 2017 vs map 2017, LVIS 2019 vs map 2019)
No stand-age screen is needed for the same-year pairs. Two filters are reported, not hidden: map no-data (water/masked) and LVIS heights > 50 m (artifacts). Results are stratified by LVIS height and canopy cover, the known drivers of ATL08-based height error (cover controls canopy photon return, and tall stands saturate the model).

In [ ]:
def same_year(df):
    d = df.copy()
    yrs = d.year.astype(int).to_numpy()
    d['pred'] = np.choose(np.searchsorted(MAP_YEARS, yrs), [d[f'ht_{y}'].to_numpy() for y in MAP_YEARS])
    d['pred_sd'] = np.choose(np.searchsorted(MAP_YEARS, yrs), [d[f'ht_sd_{y}'].to_numpy() for y in MAP_YEARS])
    d['ref'] = d.lvis_ht
    d = d[np.isfinite(d.pred) & np.isfinite(d.ref) & (d.ref <= 50)].copy()
    d['cluster_id'] = d.tile
    d['ht_class'] = pd.cut(d.ref, [0, 2, 5, 10, 15, 20, 30, 50], include_lowest=True, right=False)
    d['cc_class'] = pd.cut(d.lvis_cc, [0, 10, 30, 50, 70, 100.01], labels=['<10%', '10–30%', '30–50%', '50–70%', '≥70%'], include_lowest=True, right=False)
    return d

if RUN_LVIS:
    sy = same_year(lvis)
    m_sy = pd.concat([summarise(sy.assign(all='all'), 'all'), summarise(sy, 'year'), summarise(sy, 'cc_class'), summarise(sy, 'ht_class')])
    m_sy.to_csv(os.path.join(OUT_DIR, 'L1_lvis_same_year_metrics.csv'), index=False)
    display(m_sy.round(2))

    fig, axs = plt.subplots(1, 3, figsize=(13, 4.6), constrained_layout=True)
    for ax, (lab, g) in zip(axs, [('2017 + 2019', sy), ('LVIS 2017 vs map 2017', sy[sy.year == 2017]), ('LVIS 2019 vs map 2019', sy[sy.year == 2019])]):
        hb = hexval(ax, g.ref.values, g.pred.values, 35, lab)
        ax.set_xlabel('LVIS RH98 (30 m) [m]')
    axs[0].set_ylabel('v3.1 height, same year [m]')
    fig.colorbar(hb, ax=axs, label='pixels (log)', shrink=0.8)
    fig.savefig(os.path.join(OUT_DIR, 'FL1_lvis_same_year_hexbin.png'), dpi=200, bbox_inches='tight')

In [ ]:
if RUN_LVIS:
    fig, ax = plt.subplots(figsize=(10, 4))
    cats = sy.cc_class.cat.categories
    hcls = sy.ht_class.cat.categories
    w = 0.8 / len(cats)
    blues = ['#cde2fb', '#86b6ef', '#3987e5', '#1c5cab', '#0d366b']
    for i, c in enumerate(cats):
        data = [(g.pred - g.ref).values for _, g in sy[sy.cc_class == c].groupby('ht_class', observed=False)]
        pos = np.arange(len(hcls)) + (i - (len(cats) - 1) / 2) * w
        bp = ax.boxplot(data, positions=pos, widths=w * 0.9, showfliers=False, patch_artist=True, manage_ticks=False)
        for b in bp['boxes']:
            b.set(facecolor=blues[i], edgecolor='0.3', linewidth=0.6)
        for med in bp['medians']:
            med.set(color='0.1')
        ax.plot([], [], 's', color=blues[i], label=f'LVIS cover {c}')
    ax.axhline(0, c='0.3', lw=1)
    ax.set_xticks(np.arange(len(hcls)), [str(h) for h in hcls])
    ax.set(xlabel='LVIS RH98 class [m]', ylabel='Map − LVIS [m]', ylim=(-20, 20), title='Same-year height error by LVIS height and canopy cover')
    ax.legend(ncol=5, fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.18), frameon=False)
    fig.savefig(os.path.join(OUT_DIR, 'FL2_lvis_residual_by_height_cover.png'), dpi=200, bbox_inches='tight')

## 6. Change validation: pixels flown in both 2017 and 2019
2017 sample pixels that fall inside a 2019 LVIS tile are re-sampled from the 2019 grid. ΔLVIS = 2019 − 2017 vs Δmap = map₂₀₁₉ − map₂₀₁₇.
* Correlation and RMSD of Δ describe whether map differences carry real signal.
* **Loss detection**: a pixel is a reference loss if ΔLVIS < −`LOSS_M`. We report the precision and recall of Δmap < −`LOSS_M`.
* Caveat: two LVIS grids sampled at the same coordinates carry their own geolocation and grid noise, so ΔLVIS on stable pixels is not exactly zero. Its spread on stable pixels is reported as the reference noise floor.

In [ ]:
LOSS_M = 5
if RUN_LVIS:
    t19 = lvis_tiles[lvis_tiles.year == 2019]
    p17 = lvis[lvis.year == 2017].copy()
    rep = []
    for _, t in t19.iterrows():
        tb = gpd.GeoSeries([t.geometry], crs=t.crs).to_crs(4326).iloc[0]
        sub = p17[p17.within(tb)]
        if len(sub) == 0:
            continue
        path = lvis_files.loc[lvis_files.tile == t.tile, 'ht_path'].iloc[0]
        v = sample_raster(path, sub.geometry.x.values, sub.geometry.y.values, 'EPSG:4326', (1,), session=ornl)[:, 0]
        rep.append(sub.assign(lvis_ht_2019=v, tile_2019=t.tile))
    chg = pd.concat(rep) if rep else pd.DataFrame()
    if len(chg):
        chg = chg[np.isfinite(chg.lvis_ht_2019) & (chg.lvis_ht_2019 >= 0) & (chg.lvis_ht_2019 <= 50)]
        chg = chg[~chg.index.duplicated()]
        chg['d_ref'] = chg.lvis_ht_2019 - chg.lvis_ht
        chg['d_map'] = chg.ht_2019 - chg.ht_2017
        chg = chg[np.isfinite(chg.d_map)]
        stable = chg[np.abs(chg.d_ref) < LOSS_M]
        loss_ref, loss_map = chg.d_ref < -LOSS_M, chg.d_map < -LOSS_M
        tp = (loss_ref & loss_map).sum()
        chg_summary = pd.Series({'n_repeat_pixels': len(chg), 'r_delta': np.corrcoef(chg.d_ref, chg.d_map)[0, 1],
                                 'rmsd_delta': np.sqrt(((chg.d_map - chg.d_ref) ** 2).mean()),
                                 'ref_noise_sd_stable': stable.d_ref.std(), 'map_noise_sd_stable': stable.d_map.std(),
                                 'n_ref_loss': int(loss_ref.sum()), 'n_map_loss': int(loss_map.sum()),
                                 'loss_recall': tp / max(loss_ref.sum(), 1), 'loss_precision': tp / max(loss_map.sum(), 1)})
        chg_summary.to_csv(os.path.join(OUT_DIR, 'L2_lvis_change_2017_2019.csv'))
        print(chg_summary.round(3))
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.hexbin(chg.d_ref, chg.d_map, gridsize=40, extent=(-25, 10, -25, 10), mincnt=1, bins='log', cmap='Blues')
        ax.plot([-25, 10], [-25, 10], ls='--', c='0.3', lw=1)
        for v in (-LOSS_M,):
            ax.axvline(v, c=PAL['orange'], lw=0.8); ax.axhline(v, c=PAL['orange'], lw=0.8)
        ax.set(xlabel='ΔLVIS RH98 2019 − 2017 [m]', ylabel='Δ v3.1 height 2019 − 2017 [m]', aspect='equal',
               title=f"Change, repeat pixels (n = {len(chg):,})\nloss recall {chg_summary.loss_recall:.0%}, precision {chg_summary.loss_precision:.0%}")
        fig.savefig(os.path.join(OUT_DIR, 'FL3_lvis_change_2017_2019.png'), dpi=200, bbox_inches='tight')
    else:
        print('No 2017 sample pixels fall inside 2019 LVIS tiles.')

## 7. Temporal consistency at LVIS pixels
For pixels that are stable by the age map (2020 age > 10 yr, so no stand-replacing event since 2010), the map's inter-annual standard deviation across 2016–2025 is a noise estimate for the time series. It should be small relative to the same-year RMSD.

In [ ]:
if RUN_LVIS:
    hcols = [f'ht_{y}' for y in MAP_YEARS]
    st = lvis[(lvis.age_map > 10)].copy()
    st['ts_sd'] = st[hcols].std(axis=1)
    st['ts_mean'] = st[hcols].mean(axis=1)
    st['ht_class'] = pd.cut(st.lvis_ht, [0, 2, 5, 10, 15, 20, 30, 50], include_lowest=True, right=False)
    ts = st.groupby('ht_class', observed=True).ts_sd.describe()[['count', '25%', '50%', '75%']]
    ts.to_csv(os.path.join(OUT_DIR, 'L3_lvis_timeseries_sd_stable.csv'))
    display(ts.round(2))

## 8. G-LiHT 1 m CHM aggregated on the v3.1 grid
**Discovery first**: confirm the CMR short name and file pattern (the default `GLCHMT` and `CHM*.tif` are assumptions to check).

For each CHM granule acquired in 2016–2025:
1. find the v3.1 height tile for the acquisition year that contains it;
2. build a `WarpedVRT` of the CHM at **1 m, snapped to the map's 30 m grid** (each map pixel = exactly 30 × 30 CHM cells);
3. per map pixel: 98th percentile of the CHM (comparable to RH98) and fraction of valid cells. Keep pixels with ≥ 80 % valid cells;
4. compare to the map value for the same pixel and year. No point sampling and no grid misregistration.

In [ ]:
RUN_GLIHT = True
if RUN_GLIHT:
    import earthaccess
    earthaccess.login(strategy='netrc')
    for ds in earthaccess.search_datasets(keyword='G-LiHT canopy height model', cloud_hosted=True)[:10]:
        print(ds['umm']['ShortName'], ds['umm'].get('Version'), '|', ds['umm']['EntryTitle'])

In [ ]:
def aggregate_chm_on_map_grid(chm_path, map_path, pct=GLIHT_HT_PCT, min_valid=GLIHT_MIN_VALID_FRAC, map_bands=(1, 2), chm_session=None):
    """Per-30 m-map-pixel stats of a 1 m CHM, on the map's own grid. Returns a DataFrame (one row per map pixel)."""
    with rasterio.open(map_path) as m:
        mres = m.transform.a
        f = int(round(mres))  # CHM cells per map pixel side at 1 m
        with rasterio.Env(session=chm_session) if chm_session is not None else rasterio.Env(), rasterio.open(chm_path) as c:
            cb = rasterio.warp.transform_bounds(c.crs, m.crs, *c.bounds)
            w = rasterio.windows.from_bounds(*cb, transform=m.transform).round_offsets().round_lengths()
            try:
                w = w.intersection(Window(0, 0, m.width, m.height))
            except rasterio.errors.WindowError:
                return pd.DataFrame()
            if w.width < 1 or w.height < 1:
                return pd.DataFrame()
            wt = m.window_transform(w)
            vrt_t = rasterio.Affine(wt.a / f, 0, wt.c, 0, wt.e / f, wt.f)
            with WarpedVRT(c, crs=m.crs, transform=vrt_t, width=int(w.width) * f, height=int(w.height) * f,
                           resampling=Resampling.nearest, src_nodata=c.nodata, nodata=np.nan, dtype='float32') as v:
                a = v.read(1)
        a[(a < 0) | (a > 80)] = np.nan
        H, W = int(w.height), int(w.width)
        blocks = a.reshape(H, f, W, f).transpose(0, 2, 1, 3).reshape(H, W, f * f)
        valid = np.isfinite(blocks).mean(axis=2)
        ht = np.nanpercentile(blocks, pct, axis=2)
        cov = np.nansum(blocks >= 1.37, axis=2) / np.maximum(np.isfinite(blocks).sum(axis=2), 1)
        mv = m.read(list(map_bands), window=w).astype(float)
        if m.nodata is not None:
            mv[mv == m.nodata] = np.nan
        rr, cc = np.where(valid >= min_valid)
        xs, ys = rasterio.transform.xy(wt, rr, cc)
        return pd.DataFrame({'x': xs, 'y': ys, 'gliht_ht': ht[rr, cc], 'gliht_cc': 100 * cov[rr, cc], 'valid_frac': valid[rr, cc],
                             'pred': mv[0][rr, cc], 'pred_sd': mv[1][rr, cc] if len(map_bands) > 1 else np.nan, 'crs': str(m.crs)})

In [ ]:
if RUN_GLIHT:
    gr = earthaccess.search_data(short_name=GLIHT_SHORT_NAME, bounding_box=GLIHT_BBOX, temporal=GLIHT_TEMPORAL, count=GLIHT_MAX_GRANULES)
    print(f'{len(gr)} G-LiHT granules 2016–2025 in the ABoVE box')
    gl_parts = []
    tmp = os.path.join(OUT_DIR, 'gliht_tmp'); os.makedirs(tmp, exist_ok=True)
    fp_cache = {}
    for g in gr:
        year = int(g['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime'][:4])
        if year not in MAP_YEARS:
            continue
        links = [l for l in g.data_links() if re.search(GLIHT_CHM_PATTERN, l)]
        if not links:
            continue
        files = earthaccess.download(links[:1], tmp)
        for fn in files:
            with rasterio.open(fn) as c:
                gb = gpd.GeoSeries([box(*c.bounds)], crs=c.crs)
            fp = fp_cache.setdefault(year, gpd.read_file(HT_FOOTPRINT_FN.format(YEAR=year)))
            hit = fp[fp.intersects(gb.to_crs(fp.crs).iloc[0])]
            for mp in hit['s3_path']:
                d = aggregate_chm_on_map_grid(fn, mp)
                if len(d):
                    gl_parts.append(d.assign(year=year, granule=os.path.basename(fn)))
            os.remove(fn)
    gl = pd.concat(gl_parts, ignore_index=True) if gl_parts else pd.DataFrame()
    print(f'{len(gl):,} 30 m pixels from {gl.granule.nunique() if len(gl) else 0} granules')

In [ ]:
if RUN_GLIHT and len(gl):
    gl['ref'] = gl.gliht_ht
    gl['cluster_id'] = gl.granule
    gl = gl[np.isfinite(gl.pred)]
    gl['cc_class'] = pd.cut(gl.gliht_cc, [0, 10, 30, 50, 70, 100.01], labels=['<10%', '10–30%', '30–50%', '50–70%', '≥70%'], include_lowest=True, right=False)
    m_gl = pd.concat([summarise(gl.assign(all='all'), 'all'), summarise(gl, 'year'), summarise(gl, 'cc_class')])
    m_gl.to_csv(os.path.join(OUT_DIR, 'L4_gliht_same_year_metrics.csv'), index=False)
    display(m_gl.round(2))
    gl.to_parquet(os.path.join(OUT_DIR, 'gliht_pixels_v3.1.parquet'))
    fig, ax = plt.subplots(figsize=(5, 5))
    hb = hexval(ax, gl.ref.values, gl.pred.values, 30, 'G-LiHT CHM p98 (on map grid) vs v3.1, same year')
    ax.set(xlabel='G-LiHT 1 m CHM, p98 per 30 m pixel [m]', ylabel='v3.1 height [m]')
    fig.colorbar(hb, ax=ax, label='pixels (log)', shrink=0.8)
    fig.savefig(os.path.join(OUT_DIR, 'FL4_gliht_same_year_hexbin.png'), dpi=200, bbox_inches='tight')

## 9. Uncertainty coverage at lidar pixels
Fraction of lidar references within map mean ± 1.96 × map sd (ideal ≈ 95 %). As with the plots, the map sd excludes reference and co-registration error.

In [ ]:
rows = []
if RUN_LVIS:
    rows.append(('LVIS same-year', len(sy), 100 * np.mean(np.abs(sy.pred - sy.ref) <= 1.96 * sy.pred_sd)))
if RUN_GLIHT and len(gl):
    rows.append(('G-LiHT same-year', len(gl), 100 * np.mean(np.abs(gl.pred - gl.ref) <= 1.96 * gl.pred_sd)))
cov = pd.DataFrame(rows, columns=['reference', 'n', 'coverage_95_pct'])
cov.to_csv(os.path.join(OUT_DIR, 'L5_lidar_uncertainty_coverage.csv'), index=False)
cov